Multi-VLM classification pipeline with iterative feedback.

- *Classifier 1*: Grok 4.1 Fast
- *Classifier 2*: Qwen 3.5 Plus
- *Evaluator*: GPT-5

The evaluator checks for misclassifications and sends feedback to the classifiers. This loop runs for *5 iterations*. All models accessed via OpenRouter API.

In [ ]:
import os
import json
import base64
import shutil
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
# ================================================================
# CONFIGURATION
# ================================================================

CANDIDATE_DIR = Path("../candidate_flare_lightcurves")
OUTPUT_DIR = Path("../Final_Flares")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_ITERATIONS = 5
MAX_WORKERS = 4

# OpenRouter model IDs
CLASSIFIER_1_MODEL = "x-ai/grok-4.1-fast"        # Grok 4.1 Fast
CLASSIFIER_2_MODEL = "qwen/qwen-3.5-plus"         # Qwen 3.5 Plus
EVALUATOR_MODEL    = "openai/gpt-5"               # GPT-5

# Load API key
load_dotenv(dotenv_path="../.env")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

print(f"Candidate directory: {CANDIDATE_DIR}")
print(f"Output directory:    {OUTPUT_DIR}")
print(f"API key loaded:      {'Yes' if OPENROUTER_API_KEY else 'NO - check .env'}")

In [ ]:
# ================================================================
# HELPERS
# ================================================================

def encode_image(path: str) -> str:
    """Base64-encode an image file."""
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def call_vlm(model: str, system_prompt: str, user_text: str, image_b64: str) -> str:
    """Send an image + text to a VLM via OpenRouter and return the response text."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": user_text},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{image_b64}"
                        },
                    },
                ],
            },
        ],
        max_tokens=512,
    )
    return response.choices[0].message.content.strip()


def call_vlm_text_only(model: str, system_prompt: str, user_text: str) -> str:
    """Send a text-only prompt to a VLM via OpenRouter."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_text},
        ],
        max_tokens=1024,
    )
    return response.choices[0].message.content.strip()


def parse_classification(text: str) -> str:
    """Extract flare/non_flare label from VLM response."""
    text_lower = text.lower()
    if "non_flare" in text_lower or "non-flare" in text_lower or "no flare" in text_lower:
        return "non_flare"
    if "flare" in text_lower:
        return "flare"
    return "uncertain"


def load_candidates():
    """Load all candidate light curve images."""
    images = sorted(CANDIDATE_DIR.glob("*.png"))
    print(f"Found {len(images)} candidate light curves")
    return images

In [ ]:
# ================================================================
# PROMPTS
# ================================================================

CLASSIFIER_SYSTEM_PROMPT = """You are an expert astronomer specializing in quasar variability and transient detection.
Your task is to examine a quasar light curve plot and determine whether it contains a genuine astrophysical flare.

A flare is a significant, temporary brightening event in the light curve that stands out from the quasar's
stochastic variability (DRW noise). Look for:
- A sharp rise followed by a decay (FRED profile, Gaussian bump, or Gamma-shaped)
- Amplitude significantly above the baseline variability
- Temporal coherence (not a single-point spike which could be an artifact)

Respond ONLY in this exact format:
classification: <flare or non_flare>
confidence: <low, medium, or high>
reasoning: <one sentence explaining your decision>
"""

CLASSIFIER_USER_PROMPT = """Examine this quasar light curve and classify whether it contains a genuine flare or not."""


def get_classifier_prompt_with_feedback(feedback: str) -> str:
    """Build classifier prompt incorporating evaluator feedback."""
    return f"""{CLASSIFIER_SYSTEM_PROMPT}

IMPORTANT - Evaluator feedback from previous round:
{feedback}

Use this feedback to refine your classification. Pay close attention to the evaluator's corrections.
"""


EVALUATOR_SYSTEM_PROMPT = """You are a senior astrophysicist acting as a quality-control evaluator for a quasar flare
detection pipeline. Two independent classifiers have examined quasar light curves and provided their verdicts.

Your responsibilities:
1. Review cases where classifiers DISAGREE and determine the correct label.
2. Review cases where classifiers AGREE but with low confidence, and verify correctness.
3. Identify systematic errors (e.g., confusing artifacts/spikes with real flares, or missing subtle flares).
4. Provide actionable feedback to improve classifier accuracy in the next round.

For each light curve you review, respond in this format:
final_classification: <flare or non_flare>
confidence: <low, medium, or high>
reasoning: <brief explanation>

After reviewing all cases, provide a FEEDBACK section with guidance for the classifiers.
"""

In [ ]:
# ================================================================
# CLASSIFICATION STEP
# ================================================================

def classify_single_image(image_path: Path, system_prompt: str, model: str) -> dict:
    """Classify a single light curve image with a given model."""
    try:
        img_b64 = encode_image(str(image_path))
        response = call_vlm(model, system_prompt, CLASSIFIER_USER_PROMPT, img_b64)
        label = parse_classification(response)
        return {
            "filename": image_path.name,
            "label": label,
            "raw_response": response,
            "model": model,
        }
    except Exception as e:
        return {
            "filename": image_path.name,
            "label": "error",
            "raw_response": f"ERROR: {e}",
            "model": model,
        }


def run_classifiers(image_paths: list, system_prompt: str) -> tuple:
    """Run both classifiers on all images in parallel."""
    results_c1, results_c2 = [], []
    lock = threading.Lock()

    def classify_both(img_path):
        r1 = classify_single_image(img_path, system_prompt, CLASSIFIER_1_MODEL)
        r2 = classify_single_image(img_path, system_prompt, CLASSIFIER_2_MODEL)
        return r1, r2

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(classify_both, p): p for p in image_paths}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Classifying"):
            r1, r2 = future.result()
            with lock:
                results_c1.append(r1)
                results_c2.append(r2)

    return results_c1, results_c2

In [ ]:
# ================================================================
# EVALUATION STEP
# ================================================================

def build_evaluation_report(results_c1: list, results_c2: list) -> dict:
    """Merge classifier results and identify agreements/disagreements."""
    c1_map = {r["filename"]: r for r in results_c1}
    c2_map = {r["filename"]: r for r in results_c2}

    all_files = sorted(set(c1_map.keys()) | set(c2_map.keys()))

    merged = []
    for fn in all_files:
        r1 = c1_map.get(fn, {})
        r2 = c2_map.get(fn, {})
        l1 = r1.get("label", "missing")
        l2 = r2.get("label", "missing")

        merged.append({
            "filename": fn,
            "classifier_1_label": l1,
            "classifier_1_response": r1.get("raw_response", ""),
            "classifier_2_label": l2,
            "classifier_2_response": r2.get("raw_response", ""),
            "agreement": l1 == l2,
        })

    return merged


def run_evaluator(merged_results: list, image_paths: list) -> tuple:
    """Have the evaluator VLM review classifications and provide feedback."""
    img_map = {p.name: p for p in image_paths}

    # Prioritize disagreements and uncertain cases for evaluation
    to_review = [
        r for r in merged_results
        if not r["agreement"]
        or r["classifier_1_label"] == "uncertain"
        or r["classifier_2_label"] == "uncertain"
        or "low" in r["classifier_1_response"].lower()
        or "low" in r["classifier_2_response"].lower()
    ]

    # If all agree, still sample some for quality control
    agreed = [r for r in merged_results if r["agreement"] and r not in to_review]
    sample_size = min(5, len(agreed))
    if sample_size > 0:
        import random
        to_review.extend(random.sample(agreed, sample_size))

    evaluated_results = []

    for item in tqdm(to_review, desc="Evaluating"):
        fn = item["filename"]
        img_path = img_map.get(fn)
        if img_path is None:
            continue

        eval_prompt = (
            f"Light curve: {fn}\n"
            f"Classifier 1 (Grok 4.1 Fast): {item['classifier_1_label']}\n"
            f"  Response: {item['classifier_1_response']}\n\n"
            f"Classifier 2 (Qwen 3.5 Plus): {item['classifier_2_label']}\n"
            f"  Response: {item['classifier_2_response']}\n\n"
            f"Review this light curve image and the classifier outputs. "
            f"Determine the correct classification and explain any errors."
        )

        try:
            img_b64 = encode_image(str(img_path))
            eval_response = call_vlm(
                EVALUATOR_MODEL, EVALUATOR_SYSTEM_PROMPT, eval_prompt, img_b64
            )
            eval_label = parse_classification(eval_response)

            evaluated_results.append({
                "filename": fn,
                "evaluator_label": eval_label,
                "evaluator_response": eval_response,
            })
        except Exception as e:
            evaluated_results.append({
                "filename": fn,
                "evaluator_label": "error",
                "evaluator_response": f"ERROR: {e}",
            })

    # Generate feedback summary for classifiers
    feedback_prompt = "Summarize the evaluation results below into actionable feedback for the classifiers.\n\n"
    for er in evaluated_results:
        feedback_prompt += (
            f"- {er['filename']}: evaluator says {er['evaluator_label']}. "
            f"{er['evaluator_response'][:200]}\n"
        )
    feedback_prompt += (
        "\nProvide concise, actionable guidance for the classifiers to reduce errors "
        "in the next iteration. Focus on patterns of mistakes."
    )

    try:
        feedback = call_vlm_text_only(
            EVALUATOR_MODEL, EVALUATOR_SYSTEM_PROMPT, feedback_prompt
        )
    except Exception as e:
        feedback = f"Feedback generation failed: {e}"

    return evaluated_results, feedback

In [ ]:
# ================================================================
# MERGE FINAL LABELS
# ================================================================

def merge_final_labels(merged_results: list, evaluated_results: list) -> list:
    """Combine classifier and evaluator results into final labels.

    Priority:
    1. If evaluator reviewed the image -> use evaluator label
    2. If both classifiers agree -> use that label
    3. Otherwise -> mark as uncertain
    """
    eval_map = {r["filename"]: r for r in evaluated_results}

    final = []
    for item in merged_results:
        fn = item["filename"]
        ev = eval_map.get(fn)

        if ev and ev["evaluator_label"] not in ("error", "uncertain"):
            final_label = ev["evaluator_label"]
            source = "evaluator"
        elif item["agreement"] and item["classifier_1_label"] not in ("error", "uncertain"):
            final_label = item["classifier_1_label"]
            source = "classifier_agreement"
        else:
            final_label = "uncertain"
            source = "unresolved"

        final.append({
            "filename": fn,
            "final_label": final_label,
            "source": source,
            "classifier_1": item["classifier_1_label"],
            "classifier_2": item["classifier_2_label"],
            "evaluator": ev["evaluator_label"] if ev else "not_reviewed",
        })

    return final

In [ ]:
# ================================================================
# MAIN ITERATIVE PIPELINE
# ================================================================

image_paths = load_candidates()

feedback = ""  # No feedback for the first iteration
iteration_logs = []

for iteration in range(1, NUM_ITERATIONS + 1):
    print(f"\n{'='*60}")
    print(f"ITERATION {iteration}/{NUM_ITERATIONS}")
    print(f"{'='*60}")

    # --- Step 1: Build classifier prompt (with feedback if available) ---
    if feedback:
        current_system_prompt = get_classifier_prompt_with_feedback(feedback)
        print(f"Feedback from previous round injected ({len(feedback)} chars)")
    else:
        current_system_prompt = CLASSIFIER_SYSTEM_PROMPT
        print("No feedback yet (first iteration)")

    # --- Step 2: Run both classifiers ---
    print("\nRunning classifiers...")
    results_c1, results_c2 = run_classifiers(image_paths, current_system_prompt)

    # --- Step 3: Merge and analyze results ---
    merged = build_evaluation_report(results_c1, results_c2)
    n_agree = sum(1 for r in merged if r["agreement"])
    n_disagree = len(merged) - n_agree
    print(f"\nAgreements: {n_agree} | Disagreements: {n_disagree}")

    # --- Step 4: Evaluator reviews and generates feedback ---
    print("\nRunning evaluator (GPT-5)...")
    eval_results, feedback = run_evaluator(merged, image_paths)
    print(f"Evaluator reviewed {len(eval_results)} cases")
    print(f"\nFeedback preview: {feedback[:300]}...")

    # --- Step 5: Merge final labels ---
    final_labels = merge_final_labels(merged, eval_results)

    n_flares = sum(1 for r in final_labels if r["final_label"] == "flare")
    n_non = sum(1 for r in final_labels if r["final_label"] == "non_flare")
    n_unc = sum(1 for r in final_labels if r["final_label"] == "uncertain")
    print(f"\nResults: {n_flares} flares | {n_non} non-flares | {n_unc} uncertain")

    iteration_logs.append({
        "iteration": iteration,
        "agreements": n_agree,
        "disagreements": n_disagree,
        "flares": n_flares,
        "non_flares": n_non,
        "uncertain": n_unc,
        "evaluated": len(eval_results),
    })

print(f"\n{'='*60}")
print("ALL ITERATIONS COMPLETE")
print(f"{'='*60}")

In [ ]:
# ================================================================
# ITERATION SUMMARY
# ================================================================

log_df = pd.DataFrame(iteration_logs)
print(log_df.to_string(index=False))

In [ ]:
# ================================================================
# SAVE FINAL FLARES
# ================================================================

# Use the last iteration's final_labels (already in memory)
confirmed_flares = [r for r in final_labels if r["final_label"] == "flare"]

print(f"\nConfirmed flares after {NUM_ITERATIONS} iterations: {len(confirmed_flares)}")
print(f"Saving to: {OUTPUT_DIR}")

# Copy confirmed flare images to output folder
for r in confirmed_flares:
    src = CANDIDATE_DIR / r["filename"]
    dst = OUTPUT_DIR / r["filename"]
    if src.exists():
        shutil.copy2(src, dst)

print(f"\nCopied {len(confirmed_flares)} flare images to {OUTPUT_DIR}")
print("\nConfirmed flare files:")
for r in sorted(confirmed_flares, key=lambda x: x["filename"]):
    print(f"  {r['filename']}")